# VGAE Influence Encoder (GAIE): 3 Datasets x 3 Backbones

Runs the **GAIE** unlearning pipeline on **ML-1M, Gowalla, Yelp2018** with
**LightGCN, SimGCL, SGL** backbones (9 combinations), and prints one results table at the end.

**GAIE** encodes the influence dependency graph with a **variational graph autoencoder**: a GCN encoder produces a latent distribution q(z | A_delta) = N(mu, sigma^2); a reparameterized sample passes through an MLP shift generator to produce the embedding correction dE0; an inner-product decoder reconstructs the deleted edges, giving an ELBO-style objective (BCE reconstruction + 0.01 * KL) weighted by `rec_wei` on top of the shared L_M + lambda_u * L_u + lambda_p * L_p unlearning losses.

### Protocol (UnlearnRec, SIGIR'25, Sec. 4.1.4)
Adversarial edges (least-probable pairs under a GCN trained on clean data) are injected into the
training graph; each backbone is **trained on the attacked graph** so those edges are genuinely
learned; the unlearning target is exactly the injected edge set. This matches the original repo's
released example (`pretrain_gowalla_lightgcn_advlightgcn0.5_...`). The same GCN-generated attack
set is used for every backbone, as in the paper. MI-BF/MI-NG must exceed 1; the raw
P(before)/P(after)/P(neg) columns are the sanity check that they are not trivially satisfied.

### Runtime
Each combination = backbone pretraining (50-350 epochs) + IE unlearning + fine-tuning.
The full 3x3 will NOT fit one short GPU session — trim `RUN_COMBOS` in the config cell and run
across sessions; finished combinations are skipped via checkpoints, and results are written to
`./logs/` after **every** combination, so partial runs are never lost.

### Kaggle Setup
Enable **GPU accelerator** (Settings -> Accelerator -> GPU).

---
## 0. Environment

In [ ]:
import os
import subprocess
import torch
cuda_tag = torch.version.cuda.replace(".", "")        # e.g. "121" or "124"
torch_tag = ".".join(torch.__version__.split(".")[:2]) # e.g. "2.6"
whl_url = f"https://data.pyg.org/whl/torch-{torch_tag}.0+cu{cuda_tag}.html"
print(f"Installing torch-scatter + torch-sparse from: {whl_url}")
subprocess.check_call(["pip", "install", "-q", "torch-scatter", "torch-sparse", "-f", whl_url])
subprocess.check_call(["pip", "install", "-q", "setproctitle"])

import torch_scatter
import torch_sparse
import setproctitle
print("torch_scatter version:", torch_scatter.__version__)
print("torch_sparse version:", torch_sparse.__version__)
print("setproctitle installed \u2713")

In [ ]:
import os

REPO_URL = "https://github.com/Shuvayu12/unlearnrec_improv.git"
PROJECT_DIR = "/kaggle/working/unlearnrec_improv"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already cloned.")

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
print("Contents:", os.listdir("."))

In [ ]:
import sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Clear sys.argv so argparse in config/params.py doesn't choke on notebook kernel args
sys.argv = [sys.argv[0]]

from config.params import args
from data.data_handler import DataHandler
from Utils.time_logger import log
from Utils.utils import innerProduct, cal_mi_metrics, print_args
from models.Model import LightGCN, SimGCL, SGL, GAIE
from training.pretrain_simgcl import Coach as PretrainCoach   # dispatches on args.model
from unlearning.gaie_unlearn import Coach as IECoach

print("All imports successful!")

In [ ]:
import torch as t
import numpy as np
import random
import time
import json
import traceback

os.makedirs("./ckpt", exist_ok=True)
os.makedirs("./logs", exist_ok=True)

print(f"CUDA available: {t.cuda.is_available()}")
if t.cuda.is_available():
    print(f"GPU: {t.cuda.get_device_name(0)}")

---
## 1. Configuration

In [ ]:
PIPELINE = 'gaie'

DATASETS = ['ml1m', 'gowalla', 'yelp2018']
BACKBONES = ['lightgcn', 'simgcl', 'sgl']

# Which (dataset, backbone) pairs to run in THIS session. Completed pairs are
# skipped automatically (checkpoints + incremental JSON), so re-running is cheap.
# SESSION 1 (active): ML-1M + Gowalla, all backbones (~5-9 h, fits one 12 h session)
RUN_COMBOS = [(d, b) for d in ['ml1m', 'gowalla'] for b in BACKBONES]
# SESSION 2: RUN_COMBOS = [('yelp2018', 'lightgcn'), ('yelp2018', 'sgl')]
# SESSION 3: RUN_COMBOS = [('yelp2018', 'simgcl')]
# Full 3x3 (do NOT run in one 12 h session — a killed commit run saves no output):
# RUN_COMBOS = [(d, b) for d in DATASETS for b in BACKBONES]

DATA_CFG = {
    'ml1m':     dict(batch=2048, pretrain_epoch=50,  adv_method='lightgcn0.5',
                     pretrain_drop_rate=0.1, reg=1e-7),
    'gowalla':  dict(batch=4096, pretrain_epoch=200, adv_method='lightgcn0.5',
                     pretrain_drop_rate=0.2, reg=1e-7),
    'yelp2018': dict(batch=4096, pretrain_epoch=200, adv_method='lightgcn',
                     pretrain_drop_rate=0.1, reg=1e-6),
}

# SSL hyperparameters follow config/params.py defaults; the SimGCL values match the
# original repo's released checkpoint naming (reg1e-6_ssl1e-2_esp2e-1_t1e-1_v1 on
# Yelp2018). Not grid-searched per dataset — note this in the paper.
BACKBONE_CFG = {
    'lightgcn': dict(model='lightgcn'),
    'simgcl':   dict(model='simgcl', ssl_reg=1e-2, eps=0.2, temp=0.1, reg_version='v1'),
    'sgl':      dict(model='sgl', sgl_ssl_reg=1e-2, sgltemp=0.1, sglkeepRate=0.8),
}

# IE pre-training stage (per-dataset drop rate applied in run_pipeline)
UNLEARN_CFG = dict(
    epoch=30, lr=1e-3, batch=4096, test_drop_rate=0.003, sim_epoch=10, tst_epoch=5,
    bpr_wei=1.0, unlearn_wei=0.3, align_wei=0.1, rec_wei=0.03, align_temp=10.0,
    align_type='v2', unlearn_type='v1', overall_withdraw_rate=0.1, withdraw_rate_init=1,
    hyper_temp=1.0, unlearn_ssl=1e-3, layer_mlp=2, leaky=0.99, act='leaky',
    unlearn_layer=0, perf_degrade=0.5,
)

# Fine-tuning stage
FT_CFG = {
    'ml1m':     dict(epoch=15, unlearn_wei=0.15, align_wei=0.1, rec_wei=0.03),
    'gowalla':  dict(epoch=15, unlearn_wei=0.15, align_wei=0.1, rec_wei=0.03),
    'yelp2018': dict(epoch=15, unlearn_wei=0.15, align_wei=0.1, rec_wei=0.03),
}

# --- Per-combo retuning (session-1 results showed preservation dominating) ---
# Combos with P(before) near 1.0 (LightGCN everywhere; everything on sparse Gowalla)
# need a much stronger unlearn/preserve balance. Values follow the original repo's
# released Gowalla example (unlearn_wei 1.0 / align_wei ~0.005-0.01 / align_temp 1
# for the IE stage; 0.2 / 0.01 for fine-tuning).
STRONG_UNLEARN = dict(unlearn_wei=1.0, align_wei=0.01, align_temp=1.0)
STRONG_UNLEARN_FT = dict(unlearn_wei=0.2, align_wei=0.01)

# Paper-faithful fine-tuning (UnlearnRec Alg. 1 line 10): also train the 0-layer
# embeddings E0 during fine-tuning. Set False for the frozen-E0 (encoder-only)
# ablation, where all forgetting is attributable to the encoder architecture.
FT_TUNE_E0 = True

UNLEARN_OVERRIDES = {}
FT_COMBO_OVERRIDES = {}
for _ds in DATASETS:
    UNLEARN_OVERRIDES[(_ds, 'lightgcn')] = dict(STRONG_UNLEARN)
    FT_COMBO_OVERRIDES[(_ds, 'lightgcn')] = dict(STRONG_UNLEARN_FT)
for _bb in BACKBONES:
    UNLEARN_OVERRIDES[('gowalla', _bb)] = dict(STRONG_UNLEARN)
    FT_COMBO_OVERRIDES[('gowalla', _bb)] = dict(STRONG_UNLEARN_FT)

RESULTS_FILE = f'./logs/{PIPELINE}_3x3_results.json'
all_results = {}
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as fs:
        all_results = json.load(fs).get('results', {})
    print(f"Resuming: {len(all_results)} combination(s) already recorded")

---
## 2. Harness

`pretrain_backbone` trains (or reuses) the attacked backbone; `run_pipeline` runs
IE unlearning -> fine-tuning -> evaluation and returns one results row.

In [ ]:
def reset_seeds(seed=1234):
    t.manual_seed(seed)
    t.cuda.manual_seed_all(seed)
    t.backends.cudnn.deterministic = True
    np.random.seed(seed)
    random.seed(seed)


def apply_args(d):
    for k, v in d.items():
        setattr(args, k, v)


def set_common(ds, bb):
    cfg = DATA_CFG[ds]
    args.data = ds
    apply_args(BACKBONE_CFG[bb])
    args.adv_method = cfg['adv_method']
    args.reg = cfg['reg']
    args.seed = 1234
    args.gpu = '0'
    args.lr = 1e-3
    args.latdim = 128
    args.gnn_layer = 3
    args.topk = 20
    args.tst_bat = 256
    args.decay = 1.0
    args.bpr_wei = 1.0
    args.load_model = None
    args.adversarial_attack = True


def pretrain_backbone(ds, bb):
    set_common(ds, bb)
    cfg = DATA_CFG[ds]
    args.epoch = cfg['pretrain_epoch']
    args.batch = cfg['batch']
    args.tst_epoch = 10  # sparse pretrain evals; best-by-Recall ckpt still saved
    ckpt = f'./ckpt/pre_{ds}_{bb}_adv'
    args.save_path = ckpt
    reset_seeds()

    handler = DataHandler()
    handler.load_data(drop_rate=0.0, adv_attack=True)
    print(f"[{ds}/{bb}] users={args.user}, items={args.item}, "
          f"injected adversarial edges={len(handler.adv_edges[0])}")

    if not os.path.exists(ckpt + '.mod'):
        log(f'Pretraining {bb} on attacked {ds} graph ({args.epoch} epochs)')
        _t0 = time.time()
        PretrainCoach(handler).run()
        print(f"[{ds}/{bb}] backbone pretrain: {time.time() - _t0:.1f}s")
    else:
        print(f"[{ds}/{bb}] reusing checkpoint {ckpt}.mod")

    model = t.load(ckpt + '.mod', weights_only=False)['model'].cuda()
    model.training = False
    res = PretrainCoach(handler).tst_epoch(model)
    print(f"[{ds}/{bb}] attacked backbone Recall@{args.topk}={res['Recall']:.4f}, "
          f"NDCG@{args.topk}={res['NDCG']:.4f}")
    del handler, model
    t.cuda.empty_cache()
    return ckpt, res


def run_pipeline(ds, bb, ckpt):
    cfg = DATA_CFG[ds]

    # ---- IE unlearn (pre-training) stage ----
    set_common(ds, bb)
    apply_args(UNLEARN_CFG)
    apply_args(UNLEARN_OVERRIDES.get((ds, bb), {}))
    args.pretrain_drop_rate = cfg['pretrain_drop_rate']
    args.fineTune = False
    args.trained_model = ckpt
    args.save_path = f'./ckpt/{PIPELINE}_{ds}_{bb}'
    reset_seeds()
    h_u = DataHandler()
    h_u.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)
    coach = IECoach(h_u)
    _t0 = time.time()
    coach.run()
    t_unlearn = time.time() - _t0

    # ---- fine-tune stage ----
    args.fineTune = True
    args.ft_tune_e0 = FT_TUNE_E0
    args.model_2_finetune = args.save_path
    apply_args(FT_CFG[ds])
    apply_args(FT_COMBO_OVERRIDES.get((ds, bb), {}))
    args.save_path = f'./ckpt/{PIPELINE}_{ds}_{bb}_ft'
    reset_seeds()
    h_f = DataHandler()
    h_f.load_data(drop_rate=args.pretrain_drop_rate, adv_attack=True)
    coach = IECoach(h_f)
    _t0 = time.time()
    coach.run()
    t_ft = time.time() - _t0

    # ---- evaluate best checkpoint ----
    h_e = DataHandler()
    h_e.load_data(drop_rate=args.test_drop_rate, adv_attack=True)
    sp = args.save_path + '.mod'
    if os.path.exists(sp):
        coach.model = t.load(sp, weights_only=False)['model'].cuda()
    coach.handler = h_e
    metrics = coach.tst_epoch(coach.model)
    mi = coach.test_unlearn(coach.model, prefix=f'[{PIPELINE.upper()} {ds}/{bb}] final')

    row = dict(Recall=metrics['Recall'], NDCG=metrics['NDCG'],
               MI_BF=mi['mi_bf'], MI_NG=mi['mi_ng'],
               BeforeProb=mi['avg_before_prob'], AfterProb=mi['avg_after_prob'],
               NegProb=mi['avg_neg_prob'], UnlearnTime=t_unlearn, FinetuneTime=t_ft)
    del coach, h_u, h_f, h_e
    t.cuda.empty_cache()
    return row

---
## 3. Run all combinations

One loop; a failure in one combination is recorded and the loop continues. Results are
saved to JSON after every combination.

In [ ]:
for ds, bb in RUN_COMBOS:
    key = f'{ds}/{bb}'
    if key in all_results and 'error' not in all_results[key]:
        print(f"skip {key} (already done)")
        continue
    print('\n' + '#' * 100)
    print(f"### {PIPELINE.upper()} — {key}")
    print('#' * 100)
    try:
        ckpt, backbone_res = pretrain_backbone(ds, bb)
        row = run_pipeline(ds, bb, ckpt)
        row['BackboneRecall'] = backbone_res['Recall']
        row['BackboneNDCG'] = backbone_res['NDCG']
        all_results[key] = row
    except Exception:
        traceback.print_exc()
        all_results[key] = {'error': traceback.format_exc()[-1500:]}
    with open(RESULTS_FILE, 'w') as fs:
        json.dump({'pipeline': PIPELINE, 'seed': 1234,
                   'protocol': 'attacked-backbone (UnlearnRec Sec 4.1.4)',
                   'results': all_results}, fs, indent=2)
    print(f"saved {RESULTS_FILE}")

---
## 4. Results

In [ ]:
print('\n' + '=' * 118)
print(f"  GAIE — {len([k for k in all_results if 'error' not in all_results[k]])}/9 combinations  "
      f"(LightGCN | SimGCL | SGL backbones, attacked-graph protocol, seed 1234)")
print('=' * 118)
print(f"{'Dataset':<10} {'Backbone':<9} {'Recall@20':>10} {'NDCG@20':>9} {'MI-BF':>8} {'MI-NG':>8} "
      f"{'P(before)':>10} {'P(after)':>9} {'P(neg)':>8} {'Time(s)':>9}")
print('-' * 118)
for ds in DATASETS:
    for bb in BACKBONES:
        r = all_results.get(f'{ds}/{bb}')
        if r is None:
            print(f"{ds:<10} {bb:<9} {'not run':>10}")
            continue
        if 'error' in r:
            print(f"{ds:<10} {bb:<9} {'FAILED (see JSON)':>10}")
            continue
        tot = r['UnlearnTime'] + r['FinetuneTime']
        print(f"{ds:<10} {bb:<9} {r['Recall']:>10.4f} {r['NDCG']:>9.4f} {r['MI_BF']:>8.4f} "
              f"{r['MI_NG']:>8.4f} {r['BeforeProb']:>10.4f} {r['AfterProb']:>9.4f} "
              f"{r['NegProb']:>8.4f} {tot:>9.1f}")
    print('-' * 118)
print("Sanity: P(before) near positive level (edges were trained on); "
      "true forgetting when P(after) <= P(neg).")

# LaTeX source for the paper table
print('\n% ---- LaTeX (GAIE) ----')
print(r'\begin{tabular}{llcccc}')
print(r'\toprule')
print(r'Dataset & Backbone & Recall@20 & NDCG@20 & MI-BF & MI-NG \\')
print(r'\midrule')
for ds in DATASETS:
    for bb in BACKBONES:
        r = all_results.get(f'{ds}/{bb}')
        if r and 'error' not in r:
            print(f"{ds} & {bb} & {r['Recall']:.4f} & {r['NDCG']:.4f} & "
                  f"{r['MI_BF']:.4f} & {r['MI_NG']:.4f} \\\\")
print(r'\bottomrule')
print(r'\end{tabular}')